In [ ]:
%load_ext autoreload
%autoreload 2
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import time

import yaml
from src.sampling.main import stratified_spatial_kfold_dual

from src.utils import read_config
from src.raingauge.utils import load_raingauge_dataset, get_station_mapping_df, filter_uptime, get_station_coordinate_mappings
from benchmarks.models.idw import run_IDW_benchmark

from scipy.stats import pearsonr

In [ ]:
fold_count = 1
config_file = 'config.yaml'
with open(config_file) as f:
    config = yaml.safe_load(f)


In [ ]:
raingauge_df = load_raingauge_dataset(f'database/{config['dataset_parameters']['raingauge_file']}')

In [ ]:
raingauge_mappings = get_station_coordinate_mappings(start = 2021, end = 2025)

In [ ]:
filtered_stations = filter_uptime(raingauge_df, uptime_threshold = 0.9)
raingauge_df = raingauge_df[filtered_stations.keys()]
raingauge_df = raingauge_df.resample('15min').first() #resamples df to 15 mins
raingauge_mappings = {k:v for k, v in raingauge_mappings.items() if k in raingauge_df.keys()}

In [ ]:
#2. Get stratified training split
split_info = stratified_spatial_kfold_dual(
    raingauge_mappings, seed=123, plot=False, n_splits=fold_count
)


In [ ]:
print(raingauge_df.head(10))

In [ ]:
#3. Run the IDW for x folds
for fold in range(fold_count):
    training_gauges = split_info[fold]['statistical']['train']
    test_gauges = split_info[fold]['statistical']['test']

    # Run idw
    run_IDW_benchmark(raingauge_data=raingauge_df,
                      coordinates=raingauge_mappings,
                      training_stations=training_gauges,
                      test_stations=test_gauges,
                      power = 2,
                      n_nearest=15,
                      regression_plot=True
                      )

In [ ]:
#need to merge radar data with rain gauge data for KED

merged_df_KED = pd.merge(raingauge_choice_df, radar_df, on='time_sgt')
merged_df = raingauge_choice_df

# Kriging interpolation with rain gauge

In [ ]:
from matplotlib.colors import LogNorm

invalid_kriges = 0
training_ratio = config['dataset_parameters']['train_size']
station_names = []

for key in station_dict.keys():
  station_names.append(key,)

actual_values_arr = np.zeros(shape=[merged_df.shape[0], len(test_stations)])
predicted_values_arr = np.zeros(shape=[merged_df.shape[0], len(test_stations)])

start = time.time()

for idx in tqdm(range(len(merged_df_KED))):
  df = merged_df_KED.iloc[idx]

  kriging_result, kriging_variance = kriging_external_drift(df=df, 
                                                            station_names=training_stations, 
                                                            station_dict=station_dict, 
                                                            variogram_model='exponential', 
                                                            method='ordinary')

  if kriging_result is None:
    invalid_kriges += 1
    continue

    
  row_predicted_arr = []
  row_actual_arr = []
  
  #Calculate loss
  for test_station in test_stations:
    if not np.isnan(df[test_station]):
      rain_gauge_value = df[test_station]
      lat, lon = station_dict[test_station]
      row = math.floor((1.51 - lat) / 0.01)
      col = math.floor((lon - 103.6) / 0.01)
      kriged_value = kriging_result[row][col]

      row_actual_arr.append(rain_gauge_value)
      row_predicted_arr.append(max(0, kriged_value)) ## Kriged values can be negative and thus we need to account for this and set neg values to 0
    else:
      row_actual_arr.append(np.nan)
      row_predicted_arr.append(np.nan)
    
  actual_values_arr[idx] = np.array(row_actual_arr)
  predicted_values_arr[idx] = np.array(row_predicted_arr)



end = time.time()
MSE_arr = []

assert(len(actual_values_arr) == len(predicted_values_arr))

#calculate loss
for i in range(len(actual_values_arr)):
  pred = predicted_values_arr[i]
  act = actual_values_arr[i]
  mask = ~np.isnan(act)
  MSE = np.mean((pred[mask] - act[mask]) ** 2)
  MSE_arr.append(MSE)

average_RMSE_loss = np.sum(np.sqrt(np.array(MSE_arr))) / len(merged_df)
average_MSE_loss = np.sum(np.array(MSE_arr)) / len(merged_df)

print(f"invalid kriges: {invalid_kriges}")
print(f"final average loss: {average_RMSE_loss}")
#print(f"final average loss (0 rain = 0 loss): {total_RMSE_loss / (len(raingauge_choice_df))}")
print(f"Time taken = {end - start}")


plt.figure(figsize=(10,10))
actual = np.array(actual_values_arr).flatten()
predicted = np.array(predicted_values_arr).flatten()
mask = ~np.isnan(actual)

# plt.hist2d(x=actual[mask], y=predicted[mask],
#                   bins=100, 
#                   cmap='jet',
#                   cmin=1,
#                   norm=LogNorm(vmin=1, vmax=None))
plt.scatter(x=actual, y=predicted)

plot_bound = max(np.nanmax(actual).astype(int),np.nanmax(predicted).astype(int))
plt.plot(np.linspace(0,plot_bound,100),
        np.linspace(0,plot_bound,100))
plt.xlabel('actual_values')
plt.ylabel('predicted_values')
plt.show()


pearson_correlation, _ = pearsonr(actual[mask], predicted[mask])
print(f"Pearson correlation: {pearson_correlation}")
